**About this notebook**

This is mainly a fun experiment around a modeling idea I found interesting.

I am not claiming that the RPU is better or worse than other approaches. The goal is simply to explore how the unit works and a couple of different ways it can be used.

---

# RPU: A Flexible Relational Pattern Unit

Over a series of experiments, I have been exploring a simple way of combining a few familiar ideas for tabular modeling into one reusable unit.

I will refer to this unit as a **Relational Pattern Unit (RPU)**.

The name is mostly descriptive. The idea is to transform each feature separately, compare the transformed observation with a learned reference pattern, let a small neural network learn from those relations, and then map the resulting score to an output.

The basic structure is:

$$
\text{univariate feature shaping}
\rightarrow
\text{relational representation}
\rightarrow
\text{small multivariate network}
\rightarrow
\text{output}.
$$

The unit is not tied to one specific spline, relation function, or output function. Those parts can be changed while keeping the same general structure.

---

## 1. Univariate feature shaping

Each feature is first transformed independently:

$$
x_i \rightarrow \phi_i(x_i).
$$

The purpose is simply to give each variable its own learnable one-dimensional response before interactions between variables are introduced.

Possible choices for phi_i include:

* the raw feature itself
* piecewise linear splines
* cubic Hermite interpolation
* piecewise constant functions
* rational quadratic splines
* other one-dimensional learned transforms

In the experiments below I use cubic Hermite interpolation.

After this step:

$$
z_i = \phi_i(x_i).
$$

---

## 2. Relational representation

The transformed vector is compared with a learned reference vector

$$
c=[c_1,\ldots,c_D].
$$

Rather than immediately reducing the comparison to one scalar distance, several feature-wise or vector-valued relations can be retained.

Define

$$
d_i=z_i-c_i.
$$

Some relations I use or experiment with are the following.

### Signed squared difference

$$
\text{SignedSq}_i=d_i|d_i|.
$$

This keeps the direction of the deviation while giving larger deviations more weight.

### Element-wise product

$$
\text{Product}_i=z_ic_i.
$$

This is a simple multiplicative feature-wise relation between the transformed observation and the learned reference.

### Normalized difference

$$
\text{NormalizedDiff}_i=
\frac{z_i-c_i}
{|z_i|+|c_i|+\epsilon}.
$$

This gives a bounded signed measure of the relative difference between the transformed feature and its reference value.

### Soft ratio

First define a direct ratio:

$$
r_i=
\frac{z_i}
{c_i+\epsilon}.
$$

Since a raw ratio can become very large when \(c_i\) is close to zero, I use a compressed version:

$$
\text{SoftRatio}_i=
\frac{r_i}
{1+|r_i|}.
$$

This keeps the sign and ordering of the ratio while preventing extreme values from becoming too large.

### Projection vector

The projection of \(z\) onto the learned reference direction is

$$
\mathrm{Projection}(z,c)
=
\frac{z\cdot c}
{\|c\|^2+\epsilon}c.
$$

This produces a full vector describing the component of the transformed observation that lies along the reference direction.

### Orthogonal residual vector

The remaining component is

$$
\mathrm{Orthogonal}(z,c)
=
z-\mathrm{Projection}(z,c).
$$

Together,

$$
z
=
\mathrm{Projection}(z,c)
+
\mathrm{Orthogonal}(z,c).
$$

These relations are only examples. An RPU does not need to use all of them at once.

---

## 3. Multivariate pattern learning

The selected relation vectors are concatenated and passed to a small neural network:

$$
[
R_1(z,c),
R_2(z,c),
\ldots,
R_K(z,c)
]
\rightarrow
\text{MLP}
\rightarrow
s(x).
$$

The individual relations are constructed from the transformed features and learned reference, but the MLP sees the complete relational representation jointly.

This allows the final score to depend on combinations of several variables and several relation types at the same time.

In the experiments here, this network is intentionally small.

---

## 4. Output shaping

The scalar pattern score is passed through a learned monotonic mapping before producing the final prediction:

$$
s(x)
\rightarrow
f(s(x))
\rightarrow
\hat y.
$$

In the code I refer to this final mapping as the **fire function**.

Putting the pieces together:

$$
x
\rightarrow
\phi(x)
\rightarrow
\text{relations to a learned reference}
\rightarrow
\text{small MLP}
\rightarrow
\text{output}.
$$

---

## Why use this structure?

The main reason I find the RPU useful to experiment with is that the different parts are easy to vary.

For example, we can change:

* the univariate transformation
* the relational features
* the number of features used
* the size of the pattern network
* the output mapping

The same basic unit can also be used in different ways.

Some possible uses are:

* a single standalone predictor
* several RPUs combined in parallel
* RPU outputs used as inputs to another neural network
* randomized ensembles using different feature subsets, bootstrap samples, transforms or relations
* a weak learner inside a boosting procedure
* stacked or layered combinations of several units

There are many possible variations, and I have not explored most of them.

In this notebook I keep things simpler and look at two cases.

---

# Experiment 1 — A single strong RPU

The first experiment uses one RPU as the complete model:

$$
x
\rightarrow
RPU(x)
\rightarrow
\hat y.
$$

The unit uses the full engineered feature set with cubic Hermite transformations and three relational channels:

$$
[
\text{SignedSq},
\text{ElementwiseProduct},
\text{SoftRatio}
].
$$

These three relation vectors are concatenated and passed jointly through the small Pattern MLP.

The experiment is evaluated on the first fold of a 5-fold split and trained for up to 30 epochs with early stopping based on validation AUC.

---

# Experiment 2 — Stochastic RPU boosting with hard score regions

The second experiment uses the RPU inside a boosting-style procedure.

At each stage, a new RPU is initialized and trained briefly against the current prediction state:

$$
x
\rightarrow
RPU_t(x)
\rightarrow
s_t(x).
$$

To introduce variation between stages, each RPU receives a randomized view of the training problem.

In this experiment, each stage uses:

* a random **75–90%** subset of the training rows
* a random **65–80%** subset of the available features
* **3 randomly selected relational geometries**
* a fresh random initialization

The univariate transformation is kept fixed to cubic Hermite interpolation throughout the experiment.

The relation pool is:

$$
[
\text{SignedSq},
\text{ElementwiseProduct},
\text{NormalizedDiff},
\text{SoftRatio},
\text{Projection},
\text{OrthogonalResidual}
].
$$

The continuous RPU score is not added directly to the ensemble.

Instead, observations are ordered by the score and the one-dimensional score axis is split into a small number of hard regions:

$$
s_t(x)
\rightarrow
L_1,\ldots,L_K.
$$

For binary cross-entropy,

$$
g_i=p_i-y_i
$$

and

$$
h_i=p_i(1-p_i).
$$

Each region receives one constant Newton-style correction:

$$
v_j=
-\frac{\sum_{i\in L_j}g_i}
{\sum_{i\in L_j}h_i+\lambda}.
$$

The ensemble is updated as

$$
F_t(x)
=
F_{t-1}(x)
+
\eta v_{\ell_t(x)}.
$$

So each stage uses an expressive randomized RPU to create a nonlinear ordering, while the actual boosting contribution is restricted to a small number of constant updates.

A new randomized RPU is then trained at the next stage against the updated prediction state.

---

## Other possible experiments

There are several other ways the same basic unit could be used, but I do not test them in this notebook.

### RPU outputs as inputs to a neural network

Several RPUs could be trained in parallel:

$$
x
\rightarrow
[
RPU_1(x),
RPU_2(x),
\ldots,
RPU_M(x)
].
$$

Their scalar outputs could then be passed to an ordinary neural network:

$$
[
RPU_1(x),
\ldots,
RPU_M(x)
]
\rightarrow
\text{MLP}
\rightarrow
\hat y.
$$

The individual RPUs could use different feature subsets, univariate transforms, relation sets, knot counts, internal widths, or random initializations.

---

### A randomized RPU ensemble

Another possibility is a more bagging- or random-forest-like construction.

Each RPU could use its own randomized setup, for example:

* a different bootstrap sample of rows
* a different random feature subset
* a different univariate transform
* a different subset of relational geometries
* different knot counts
* different internal widths
* different random seeds

The resulting predictions could then simply be averaged:

$$
\hat p(x)
=
\frac{1}{M}
\sum_{j=1}^{M}
\hat p_j(x).
$$

The idea would simply be to create diversity between the units through both the data they see and the internal RPU configuration they use.

These are possible extensions of the same basic idea, but the experiments below focus only on the standalone and boosting cases.


## Imports

In [ ]:
import os
import glob
import math
import copy
import random
import warnings

import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import TargetEncoder
from sklearn.metrics import roc_auc_score

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

warnings.filterwarnings("ignore")

**Compute note:** The experiments in this notebook were run on CPU. The code automatically uses CUDA when a compatible GPU is available, which should make training considerably faster.

## Experiment 1 — Single Strong RPU

We start with a single RPU used as the complete model.

For this experiment, the RPU uses the full engineered feature set, cubic Hermite feature transformations, and three relational channels: **SignedSq, ElementwiseProduct, and SoftRatio**.

The model is evaluated on the first fold of a 5-fold split and trained for up to 30 epochs with early stopping based on validation AUC.


In [ ]:
# Experiment 1: Single Strong RPU — Fold 1
SEED = 21
ID = "id"
TARGET = "Will_Buy_EV"
N_SPLITS = 5
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

RAW_NUMS = [
    "Age",
    "Annual_Income_USD",
    "Daily_Commute_km",
    "Number_of_Cars_Owned",
    "Charging_Stations_Near_Home",
    "Charging_Stations_Near_Work",
    "Environmental_Concern_Level",
]
RAW_CATS = [
    "Gender",
    "City_Type",
    "Current_Car_Type",
    "Home_Charging_Possible",
    "Subsidy_Available",
    "Range_Anxiety_Level",
]
DIGIT_KS = list(range(-4, 4))

# Configuration
PATTERN_HIDDEN = 8
N_FIRE_KNOTS = 8
HERMITE_KNOTS = 15
GLOBAL_SPLINE_MIN = -8.0
GLOBAL_SPLINE_MAX = 8.0
LAMBDA_HERMITE_SMOOTH = 1e-6
LAMBDA_FIRE_SMOOTH = 1e-5

MAX_EPOCHS = 30
PATIENCE = 6
BATCH_SIZE = 2048
PRED_BATCH_SIZE = 8192
LR = 1.5e-3
WEIGHT_DECAY = 2e-5
GRAD_CLIP = 5.0

OUTPUT_DIR = "/kaggle/working"
os.makedirs(OUTPUT_DIR, exist_ok=True)

os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.benchmark = False

print("Device:", DEVICE)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

# Load data

competition_roots = [
    "/kaggle/input/competitions/playground-series-s6e9",
    "/kaggle/input/playground-series-s6e9",
]

train_path = None
test_path = None
sample_path = None

for root in competition_roots:
    candidate_train = os.path.join(root, "train.csv")
    candidate_test = os.path.join(root, "test.csv")
    candidate_sample = os.path.join(root, "sample_submission.csv")

    if os.path.exists(candidate_train) and os.path.exists(candidate_test):
        train_path = candidate_train
        test_path = candidate_test
        sample_path = candidate_sample if os.path.exists(candidate_sample) else None
        break

if train_path is None:
    candidates = [
        p for p in glob.glob("/kaggle/input/**/train.csv", recursive=True)
        if "playground-series-s6e9" in p.lower()
    ]
    if not candidates:
        raise FileNotFoundError("Could not locate the S6E9 competition files.")

    train_path = candidates[0]
    root = os.path.dirname(train_path)
    test_path = os.path.join(root, "test.csv")
    sample_path = os.path.join(root, "sample_submission.csv")

train = pd.read_csv(train_path)
test = pd.read_csv(test_path)

# Convert the target robustly whether Kaggle stores it as Yes/No or already as 0/1.
if not pd.api.types.is_numeric_dtype(train[TARGET]):
    train[TARGET] = (
        train[TARGET]
        .astype(str)
        .str.strip()
        .str.lower()
        .map({
            "no": 0.0,
            "yes": 1.0,
            "0": 0.0,
            "1": 1.0,
            "false": 0.0,
            "true": 1.0,
        })
    )
else:
    train[TARGET] = pd.to_numeric(train[TARGET], errors="coerce")

if train[TARGET].isna().any():
    raise ValueError("Target conversion produced missing values.")

train[TARGET] = train[TARGET].astype(np.float32)
y = train[TARGET].to_numpy(dtype=np.float32)

print("Train:", train.shape)
print("Test :", test.shape)
print("Positive rate:", float(y.mean()))


# Fold-safe feature engineering

def safe_numeric(series):
    return pd.to_numeric(series, errors="coerce").astype(np.float64)


def stringify_values(values):
    s = pd.Series(values, copy=False).astype("object")
    s = s.where(~pd.isna(s), "__NA__")
    return s.astype(str).to_numpy(dtype=object)


def build_base_frame(df):
    """Build the base numerical feature block."""
    out = pd.DataFrame(index=np.arange(len(df)))

    for col in RAW_NUMS:
        out[col] = safe_numeric(df[col]).to_numpy()

    for col in RAW_NUMS:
        x = safe_numeric(df[col]).to_numpy()

        for k in DIGIT_KS:
            scale = 10.0 ** k
            digit = np.floor(x / scale)
            digit = np.mod(digit, 10.0)
            out[f"{col}__digit_k{k}"] = digit

    income = safe_numeric(df["Annual_Income_USD"]).to_numpy()
    concern = safe_numeric(df["Environmental_Concern_Level"]).to_numpy()

    out["flag_income_eq_30000"] = (income == 30000).astype(np.float64)
    out["flag_income_ge_170537"] = (income >= 170537).astype(np.float64)
    out["flag_income_38k_42k"] = (
        (income >= 38000) & (income <= 42000)
    ).astype(np.float64)
    out["flag_env_concern_eq_1"] = (concern == 1).astype(np.float64)

    return out


def fill_base_from_train(train_base, valid_base, test_base):
    for col in train_base.columns:
        median = np.nanmedian(train_base[col].to_numpy(dtype=np.float64))

        if not np.isfinite(median):
            median = 0.0

        train_base[col] = train_base[col].fillna(median)
        valid_base[col] = valid_base[col].fillna(median)
        test_base[col] = test_base[col].fillna(median)


def source_values(raw_df, base_df, source_name):
    kind, col = source_name.split("::", 1)

    if kind == "rawcat":
        return stringify_values(raw_df[col].to_numpy())

    if kind == "numcat":
        return stringify_values(safe_numeric(raw_df[col]).to_numpy())

    if kind == "digitcat":
        return stringify_values(base_df[col].to_numpy())

    raise ValueError(source_name)


def make_single_col_matrix(values):
    return np.asarray(values, dtype=object).reshape(-1, 1)


def target_encode_builtin(
    train_values,
    valid_values,
    test_values,
    y_train,
    smooth,
    seed,
):
    """Fold-safe target encoding."""
    encoder = TargetEncoder(
        smooth=smooth,
        cv=5,
        shuffle=True,
        random_state=seed,
        target_type="binary",
    )

    train_encoded = encoder.fit_transform(
        make_single_col_matrix(train_values),
        y_train,
    ).reshape(-1)

    valid_encoded = encoder.transform(
        make_single_col_matrix(valid_values)
    ).reshape(-1)

    test_encoded = encoder.transform(
        make_single_col_matrix(test_values)
    ).reshape(-1)

    return train_encoded, valid_encoded, test_encoded


def build_fold_features(train_raw, test_raw, train_idx, valid_idx, fold_number):
    """
    Rebuild the complete record representation independently for one outer fold.

    Expected Fold-1 accounting:
        67 base
      + 69 frequency
      + 69 TE(auto)
      + 69 TE(10)
      = 274 numerical candidates

    Historical constant removal:
        274 - 76 = 198 numerical features

    Plus 6 raw categorical IDs:
        198 + 6 = 204 final features
    """
    fold_train = train_raw.iloc[train_idx].reset_index(drop=True)
    fold_valid = train_raw.iloc[valid_idx].reset_index(drop=True)

    y_train = fold_train[TARGET].to_numpy(dtype=np.float32)
    y_valid = fold_valid[TARGET].to_numpy(dtype=np.float32)

    base_train = build_base_frame(fold_train)
    base_valid = build_base_frame(fold_valid)
    base_test = build_base_frame(test_raw)

    fill_base_from_train(base_train, base_valid, base_test)

    digit_cols = [
        col for col in base_train.columns
        if "__digit_k" in col
    ]

    encoding_sources = (
        [f"rawcat::{col}" for col in RAW_CATS]
        + [f"numcat::{col}" for col in RAW_NUMS]
        + [f"digitcat::{col}" for col in digit_cols]
    )

    assert len(encoding_sources) == 69

    # ------------------------------------------------------------------
    # Frequency encoding
    # ------------------------------------------------------------------

    freq_train = pd.DataFrame(index=np.arange(len(fold_train)))
    freq_valid = pd.DataFrame(index=np.arange(len(fold_valid)))
    freq_test = pd.DataFrame(index=np.arange(len(test_raw)))

    for source in encoding_sources:
        train_values = source_values(fold_train, base_train, source)
        valid_values = source_values(fold_valid, base_valid, source)
        test_values = source_values(test_raw, base_test, source)

        frequency_map = (
            pd.Series(train_values)
            .value_counts(dropna=False, normalize=True)
            .to_dict()
        )

        name = f"freq::{source}"

        freq_train[name] = (
            pd.Series(train_values)
            .map(frequency_map)
            .fillna(0.0)
            .to_numpy(np.float64)
        )

        freq_valid[name] = (
            pd.Series(valid_values)
            .map(frequency_map)
            .fillna(0.0)
            .to_numpy(np.float64)
        )

        freq_test[name] = (
            pd.Series(test_values)
            .map(frequency_map)
            .fillna(0.0)
            .to_numpy(np.float64)
        )

    # ------------------------------------------------------------------
    # Raw categorical IDs
    # ------------------------------------------------------------------

    catid_train = pd.DataFrame(index=np.arange(len(fold_train)))
    catid_valid = pd.DataFrame(index=np.arange(len(fold_valid)))
    catid_test = pd.DataFrame(index=np.arange(len(test_raw)))

    for col in RAW_CATS:
        train_values = stringify_values(fold_train[col].to_numpy())
        valid_values = stringify_values(fold_valid[col].to_numpy())
        test_values = stringify_values(test_raw[col].to_numpy())

        unique_values = pd.Index(pd.unique(train_values))
        mapping = {
            value: i + 1
            for i, value in enumerate(unique_values)
        }

        name = f"catid::{col}"

        catid_train[name] = (
            pd.Series(train_values)
            .map(mapping)
            .fillna(0)
            .astype(np.float64)
            .to_numpy()
        )

        catid_valid[name] = (
            pd.Series(valid_values)
            .map(mapping)
            .fillna(0)
            .astype(np.float64)
            .to_numpy()
        )

        catid_test[name] = (
            pd.Series(test_values)
            .map(mapping)
            .fillna(0)
            .astype(np.float64)
            .to_numpy()
        )

    # ------------------------------------------------------------------
    # Two historical target-encoding blocks
    # ------------------------------------------------------------------

    te_auto_train = pd.DataFrame(index=np.arange(len(fold_train)))
    te_auto_valid = pd.DataFrame(index=np.arange(len(fold_valid)))
    te_auto_test = pd.DataFrame(index=np.arange(len(test_raw)))

    te_10_train = pd.DataFrame(index=np.arange(len(fold_train)))
    te_10_valid = pd.DataFrame(index=np.arange(len(fold_valid)))
    te_10_test = pd.DataFrame(index=np.arange(len(test_raw)))

    # Fold 1 uses seed 21, matching the historical fold seed.
    encoder_seed = SEED + (fold_number - 1)

    print(f"Fold {fold_number}: building 2 x 69 target encodings...")

    for j, source in enumerate(encoding_sources):
        train_values = source_values(fold_train, base_train, source)
        valid_values = source_values(fold_valid, base_valid, source)
        test_values = source_values(test_raw, base_test, source)

        auto_train, auto_valid, auto_test = target_encode_builtin(
            train_values,
            valid_values,
            test_values,
            y_train,
            smooth="auto",
            seed=encoder_seed,
        )

        ten_train, ten_valid, ten_test = target_encode_builtin(
            train_values,
            valid_values,
            test_values,
            y_train,
            smooth=10.0,
            seed=encoder_seed,
        )

        te_auto_train[f"te_auto::{source}"] = auto_train
        te_auto_valid[f"te_auto::{source}"] = auto_valid
        te_auto_test[f"te_auto::{source}"] = auto_test

        te_10_train[f"te_10::{source}"] = ten_train
        te_10_valid[f"te_10::{source}"] = ten_valid
        te_10_test[f"te_10::{source}"] = ten_test

        if (j + 1) % 10 == 0 or j + 1 == len(encoding_sources):
            print(f"  encoded {j + 1}/{len(encoding_sources)} sources")

    numeric_train = pd.concat(
        [
            base_train.reset_index(drop=True),
            freq_train,
            te_auto_train,
            te_10_train,
        ],
        axis=1,
    )

    numeric_valid = pd.concat(
        [
            base_valid.reset_index(drop=True),
            freq_valid,
            te_auto_valid,
            te_10_valid,
        ],
        axis=1,
    )

    numeric_test = pd.concat(
        [
            base_test.reset_index(drop=True),
            freq_test,
            te_auto_test,
            te_10_test,
        ],
        axis=1,
    )

    assert numeric_train.shape[1] == 274, numeric_train.shape

    numeric_train = numeric_train.replace([np.inf, -np.inf], np.nan)
    numeric_valid = numeric_valid.replace([np.inf, -np.inf], np.nan)
    numeric_test = numeric_test.replace([np.inf, -np.inf], np.nan)

    # Final train-only median imputation.
    for col in numeric_train.columns:
        median = np.nanmedian(
            numeric_train[col].to_numpy(dtype=np.float64)
        )

        if not np.isfinite(median):
            median = 0.0

        numeric_train[col] = numeric_train[col].fillna(median)
        numeric_valid[col] = numeric_valid[col].fillna(median)
        numeric_test[col] = numeric_test[col].fillna(median)

    # Remove numerical columns that are constant in this fold's training rows.
    unique_counts = numeric_train.nunique(dropna=False)

    keep_numeric_cols = unique_counts[
        unique_counts > 1
    ].index.tolist()

    dropped_numeric_cols = unique_counts[
        unique_counts <= 1
    ].index.tolist()

    numeric_train = numeric_train[keep_numeric_cols]
    numeric_valid = numeric_valid[keep_numeric_cols]
    numeric_test = numeric_test[keep_numeric_cols]

    full_train = pd.concat(
        [
            numeric_train.reset_index(drop=True),
            catid_train,
        ],
        axis=1,
    )

    full_valid = pd.concat(
        [
            numeric_valid.reset_index(drop=True),
            catid_valid,
        ],
        axis=1,
    )

    full_test = pd.concat(
        [
            numeric_test.reset_index(drop=True),
            catid_test,
        ],
        axis=1,
    )

    feature_names = full_train.columns.tolist()

    print(
        f"Fold {fold_number}: "
        f"{len(keep_numeric_cols)} numerical + "
        f"{catid_train.shape[1]} categorical IDs = "
        f"{len(feature_names)} final features"
    )

    if fold_number == 1:
        print(
            "Historical Fold-1 target: "
            "198 numerical + 6 categorical IDs = 204."
        )

        if len(feature_names) != 204:
            print(
                "WARNING: Fold 1 did not reproduce 204 features. "
                "Do not treat this run as an exact record-feature reproduction."
            )

        print(
            f"Fold 1 numerical constants removed: "
            f"{len(dropped_numeric_cols)}"
        )

    x_train_raw = full_train.to_numpy(dtype=np.float32)
    x_valid_raw = full_valid.to_numpy(dtype=np.float32)
    x_test_raw = full_test.to_numpy(dtype=np.float32)

    # Standardize from outer-fold training rows only.
    mean = x_train_raw.mean(
        axis=0,
        dtype=np.float64,
    ).astype(np.float32)

    std = x_train_raw.std(
        axis=0,
        dtype=np.float64,
    ).astype(np.float32)

    mean = np.where(
        np.isfinite(mean),
        mean,
        0.0,
    ).astype(np.float32)

    std = np.where(
        np.isfinite(std) & (std > 1e-7),
        std,
        1.0,
    ).astype(np.float32)

    x_train = np.clip(
        (x_train_raw - mean) / std,
        -8.0,
        8.0,
    ).astype(np.float32)

    x_valid = np.clip(
        (x_valid_raw - mean) / std,
        -8.0,
        8.0,
    ).astype(np.float32)

    x_test = np.clip(
        (x_test_raw - mean) / std,
        -8.0,
        8.0,
    ).astype(np.float32)

    metadata = pd.DataFrame({
        "feature_name": feature_names,
        "mean": mean,
        "std": std,
    })

    return (
        x_train,
        y_train,
        x_valid,
        y_valid,
        x_test,
        feature_names,
        metadata,
    )

class StrongRPU(nn.Module):
    """One compact all-feature RPU with learned univariate cubic Hermite transforms."""

    def __init__(self, input_dim, target_rate):
        super().__init__()
        self.input_dim = int(input_dim)
        self.joint_dim = 3 * self.input_dim

        # Fixed equally spaced x-knots over the standardized feature range.
        # Each feature learns its own y-value at every knot. Hermite interpolation
        # makes the transform C1-smooth between knots.
        knot_x = torch.linspace(GLOBAL_SPLINE_MIN, GLOBAL_SPLINE_MAX, HERMITE_KNOTS)
        self.register_buffer("knot_x", knot_x)

        # Identity initialization: s_i(x) = x at the start of training.
        self.spline_y = nn.Parameter(
            knot_x.unsqueeze(0).repeat(self.input_dim, 1).clone()
        )

        # Learned reference point in transformed feature space.
        self.center = nn.Parameter(torch.randn(self.input_dim) * 0.15)

        # One shared Pattern MLP over SignedSq, ElementwiseProduct, and SoftRatio jointly.
        scale1 = 0.20 / math.sqrt(max(self.joint_dim, 1))
        scale2 = 0.20 / math.sqrt(max(PATTERN_HIDDEN, 1))
        self.pattern_w1 = nn.Parameter(torch.randn(PATTERN_HIDDEN, self.joint_dim) * scale1)
        self.pattern_b1 = nn.Parameter(torch.zeros(PATTERN_HIDDEN))
        self.pattern_w2 = nn.Parameter(torch.randn(PATTERN_HIDDEN) * scale2)
        self.pattern_b2 = nn.Parameter(torch.zeros(()))

        # Monotonic calibration from Pattern score to fire strength.
        inv_softplus_one = math.log(math.expm1(1.0))
        self.raw_fire_increments = nn.Parameter(
            torch.full((N_FIRE_KNOTS - 1,), inv_softplus_one)
        )

        # Final scale/bias turns fire strength into a binary logit.
        self.raw_prediction_scale = nn.Parameter(torch.tensor(math.log(math.expm1(2.0))))
        p = float(np.clip(target_rate, 1e-5, 1.0 - 1e-5))
        self.prediction_bias = nn.Parameter(torch.tensor(math.log(p / (1.0 - p))))

    def spline_transform(self, x):
        # Local cubic Hermite interpolation. Knot y-values are learned and slopes
        # are estimated from neighboring learned knots, keeping the transform local.
        lo = float(self.knot_x[0])
        hi = float(self.knot_x[-1])
        n = self.knot_x.numel()
        xc = x.clamp(lo, hi)
        pos = (xc - lo) / max(hi - lo, 1e-8) * (n - 1)
        i0 = torch.floor(pos).long().clamp(0, n - 2)
        i1 = i0 + 1
        t = pos - i0.to(pos.dtype)

        h = (hi - lo) / max(n - 1, 1)
        y = self.spline_y
        m = torch.empty_like(y)
        m[:, 0] = (y[:, 1] - y[:, 0]) / h
        m[:, -1] = (y[:, -1] - y[:, -2]) / h
        m[:, 1:-1] = (y[:, 2:] - y[:, :-2]) / (2.0 * h)

        ytab = y.unsqueeze(0).expand(x.shape[0], -1, -1)
        mtab = m.unsqueeze(0).expand(x.shape[0], -1, -1)
        y0 = torch.gather(ytab, 2, i0.unsqueeze(2)).squeeze(2)
        y1 = torch.gather(ytab, 2, i1.unsqueeze(2)).squeeze(2)
        m0 = torch.gather(mtab, 2, i0.unsqueeze(2)).squeeze(2)
        m1 = torch.gather(mtab, 2, i1.unsqueeze(2)).squeeze(2)

        t2 = t * t
        t3 = t2 * t
        h00 = 2.0 * t3 - 3.0 * t2 + 1.0
        h10 = t3 - 2.0 * t2 + t
        h01 = -2.0 * t3 + 3.0 * t2
        h11 = t3 - t2
        return h00 * y0 + h10 * h * m0 + h01 * y1 + h11 * h * m1

    def joint_pattern_score(self, x):
        xt = self.spline_transform(x)
        c = self.center.unsqueeze(0)
        diff = xt - c

        signed_sq = diff * diff.abs()
        product = xt * c
        raw_ratio = xt / (c + 1e-4)
        soft_ratio = raw_ratio / (1.0 + raw_ratio.abs())

        relations = torch.cat([signed_sq, product, soft_ratio], dim=1)
        hidden = F.gelu(relations @ self.pattern_w1.t() + self.pattern_b1)
        logit = hidden @ self.pattern_w2 + self.pattern_b2
        return torch.sigmoid(logit)

    def fire_values(self):
        inc = F.softplus(self.raw_fire_increments) + 1e-6
        cum = torch.cumsum(inc, dim=0)
        cum = cum / cum[-1].clamp_min(1e-8)
        return torch.cat([torch.zeros(1, device=cum.device, dtype=cum.dtype), cum])

    def apply_fire_curve(self, score):
        pos = score.clamp(0.0, 1.0) * (N_FIRE_KNOTS - 1)
        idx0 = torch.floor(pos).long().clamp(0, N_FIRE_KNOTS - 2)
        frac = pos - idx0.to(pos.dtype)
        values = self.fire_values()
        return values[idx0] + frac * (values[idx0 + 1] - values[idx0])

    def forward(self, x):
        score = self.apply_fire_curve(self.joint_pattern_score(x))
        scale = F.softplus(self.raw_prediction_scale) + 1e-4
        return self.prediction_bias + scale * (score - 0.5)

    def regularization(self):
        spline_d2 = self.spline_y[:, 2:] - 2.0 * self.spline_y[:, 1:-1] + self.spline_y[:, :-2]
        fire = self.fire_values()
        fire_d2 = fire[2:] - 2.0 * fire[1:-1] + fire[:-2]
        return (
            LAMBDA_HERMITE_SMOOTH * spline_d2.pow(2).mean()
            + LAMBDA_FIRE_SMOOTH * fire_d2.pow(2).mean()
        )

    @torch.no_grad()
    def initialize_center(self, x_train, y_train, seed):
        rng = np.random.default_rng(seed)
        positive_rows = np.flatnonzero(y_train >= 0.5)
        pool = positive_rows if len(positive_rows) else np.arange(len(y_train))
        idx = int(rng.choice(pool))
        row = torch.from_numpy(x_train[idx:idx + 1]).to(self.center.device, self.center.dtype)
        self.center.copy_(self.spline_transform(row).squeeze(0))



class BinaryDataset(Dataset):
    def __init__(self, x, y):
        self.x = x
        self.y = y.astype(np.float32)

    def __len__(self):
        return len(self.x)

    def __getitem__(self, i):
        return torch.from_numpy(self.x[i]), torch.tensor(self.y[i], dtype=torch.float32)


@torch.no_grad()
def predict_logits(model, x, batch_size=PRED_BATCH_SIZE):
    model.eval()
    out = np.empty(len(x), dtype=np.float32)
    for start in range(0, len(x), batch_size):
        end = min(start + batch_size, len(x))
        xb = torch.from_numpy(x[start:end]).to(DEVICE, non_blocking=True)
        out[start:end] = model(xb).float().cpu().numpy()
    return out


def sigmoid_np(x):
    x = np.clip(x, -30.0, 30.0)
    return 1.0 / (1.0 + np.exp(-x))


def train_fold1(train_raw, test_raw, train_idx, valid_idx):
    print("\n" + "=" * 120)
    print(f"SINGLE STRONG RPU | FOLD 1/{N_SPLITS} | train={len(train_idx):,} | valid={len(valid_idx):,}")
    print("=" * 120)

    x_train, y_train, x_valid, y_valid, x_test, feature_names, feature_metadata = build_fold_features(
        train_raw, test_raw, train_idx, valid_idx, fold_number=1
    )

    model_seed = SEED
    center_seed = SEED + 8100
    random.seed(model_seed)
    np.random.seed(model_seed)
    torch.manual_seed(model_seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(model_seed)
        torch.cuda.manual_seed_all(model_seed)

    model = StrongRPU(
        input_dim=x_train.shape[1],
        target_rate=float(y_train.mean()),
    ).to(DEVICE)
    model.initialize_center(x_train, y_train, center_seed)

    print(f"Features used          : {x_train.shape[1]} / {x_train.shape[1]}")
    print(f"Hermite knots / feature    : {HERMITE_KNOTS}")
    print(f"Joint Pattern dim      : {3 * x_train.shape[1]}")

    loader = DataLoader(
        BinaryDataset(x_train, y_train),
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=0,
        pin_memory=(DEVICE.type == "cuda"),
        drop_last=False,
    )

    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=MAX_EPOCHS, eta_min=LR * 0.12
    )
    criterion = nn.BCEWithLogitsLoss()

    best_auc = -np.inf
    best_state = None
    best_epoch = 0
    stale = 0
    history = []

    for epoch in range(1, MAX_EPOCHS + 1):
        model.train()
        loss_sum = 0.0
        n_seen = 0

        for xb, yb in loader:
            xb = xb.to(DEVICE, non_blocking=True)
            yb = yb.to(DEVICE, non_blocking=True)

            optimizer.zero_grad(set_to_none=True)
            logits = model(xb)
            data_loss = criterion(logits, yb)
            loss = data_loss + model.regularization()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
            optimizer.step()

            bs = len(xb)
            loss_sum += float(data_loss.detach().cpu()) * bs
            n_seen += bs

        scheduler.step()

        valid_logits = predict_logits(model, x_valid)
        valid_prob = sigmoid_np(valid_logits)
        valid_auc = float(roc_auc_score(y_valid, valid_prob))
        valid_bce = float(F.binary_cross_entropy_with_logits(
            torch.from_numpy(valid_logits),
            torch.from_numpy(y_valid.astype(np.float32)),
        ).item())

        improved = valid_auc > best_auc + 1e-7
        if improved:
            best_auc = valid_auc
            best_state = copy.deepcopy(model.state_dict())
            best_epoch = epoch
            stale = 0
        else:
            stale += 1

        row = {
            "epoch": epoch,
            "train_bce": loss_sum / max(n_seen, 1),
            "valid_bce": valid_bce,
            "valid_auc": valid_auc,
        }
        history.append(row)
        marker = " *BEST*" if improved else ""
        print(
            f"Epoch {epoch:02d}/{MAX_EPOCHS} | "
            f"trainBCE={row['train_bce']:.6f} | validBCE={valid_bce:.6f} | "
            f"AUC={valid_auc:.8f}{marker}"
        )

        if stale >= PATIENCE:
            print(f"Early stopping at epoch {epoch}.")
            break

    if best_state is None:
        raise RuntimeError("No checkpoint created")

    model.load_state_dict(best_state)
    valid_prob = sigmoid_np(predict_logits(model, x_valid))
    final_auc = float(roc_auc_score(y_valid, valid_prob))

    print("-" * 120)
    print(f"BEST EPOCH     : {best_epoch}")
    print(f"VALID AUC      : {final_auc:.8f}")
    print(f"FEATURES       : {x_train.shape[1]}")
    print(f"PATTERN INPUT  : {3 * x_train.shape[1]}")
    print("-" * 120)

    return final_auc



outer_cv = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
train_idx, valid_idx = next(outer_cv.split(np.zeros(len(train)), y))
auc = train_fold1(
    train, test, train_idx, valid_idx
)

print("\n" + "=" * 120)
print("EXPERIMENT 1 — SINGLE STRONG RPU COMPLETE")
print(f"Validation AUC: {auc:.8f}")
print("=" * 120)


## Experiment 2 — Stochastic RPU Boosting

In the second experiment, the RPU is used as a weak learner inside a boosting-style procedure.

A new RPU is initialized at each stage and trained only briefly on the current prediction state. To encourage variation between stages, each learner receives a randomized view of the data:

* roughly **65–80% of the features**
* roughly **75–90% of the training rows**
* **3 randomly selected relational geometries**
* a fresh random initialization

The available relation pool includes **SignedSq, ElementwiseProduct, NormalizedDiff, SoftRatio, Projection**, and **OrthogonalResidual**.

Each RPU produces a continuous scalar score, but that score is not added directly to the ensemble. Instead, the score is split into a small number of hard regions, and each region receives a constant Newton-style correction.

Conceptually:

$$
\text{randomized RPU}
\rightarrow
\text{scalar score}
\rightarrow
\text{hard score regions}
\rightarrow
\text{Newton update}.
$$

A new randomized RPU is then trained at the next stage against the updated prediction state.


In [ ]:
# Experiment 2: Stochastic RPU Boosting — Fold 1

SEED = 21
ID = "id"
TARGET = "Will_Buy_EV"
N_SPLITS = 5
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

RAW_NUMS = [
    "Age",
    "Annual_Income_USD",
    "Daily_Commute_km",
    "Number_of_Cars_Owned",
    "Charging_Stations_Near_Home",
    "Charging_Stations_Near_Work",
    "Environmental_Concern_Level",
]
RAW_CATS = [
    "Gender",
    "City_Type",
    "Current_Car_Type",
    "Home_Charging_Possible",
    "Subsidy_Available",
    "Range_Anxiety_Level",
]
DIGIT_KS = list(range(-4, 4))

# Model/training config
PATTERN_HIDDEN = 8
N_FIRE_KNOTS = 8
HERMITE_KNOTS = 15
GLOBAL_SPLINE_MIN = -8.0
GLOBAL_SPLINE_MAX = 8.0
LAMBDA_HERMITE_SMOOTH = 1e-6
LAMBDA_FIRE_SMOOTH = 1e-5

BATCH_SIZE = 2048
PRED_BATCH_SIZE = 8192
GRAD_CLIP = 5.0

OUTPUT_DIR = "/kaggle/working"
os.makedirs(OUTPUT_DIR, exist_ok=True)

os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.benchmark = False

print("Device:", DEVICE)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

# ------------------------------------------------------------------------------------------
# Load competition data
# ------------------------------------------------------------------------------------------

competition_roots = [
    "/kaggle/input/competitions/playground-series-s6e9",
    "/kaggle/input/playground-series-s6e9",
]

train_path = None
test_path = None
sample_path = None

for root in competition_roots:
    candidate_train = os.path.join(root, "train.csv")
    candidate_test = os.path.join(root, "test.csv")
    candidate_sample = os.path.join(root, "sample_submission.csv")

    if os.path.exists(candidate_train) and os.path.exists(candidate_test):
        train_path = candidate_train
        test_path = candidate_test
        sample_path = candidate_sample if os.path.exists(candidate_sample) else None
        break

if train_path is None:
    candidates = [
        p for p in glob.glob("/kaggle/input/**/train.csv", recursive=True)
        if "playground-series-s6e9" in p.lower()
    ]
    if not candidates:
        raise FileNotFoundError("Could not locate the S6E9 competition files.")

    train_path = candidates[0]
    root = os.path.dirname(train_path)
    test_path = os.path.join(root, "test.csv")
    sample_path = os.path.join(root, "sample_submission.csv")

train = pd.read_csv(train_path)
test = pd.read_csv(test_path)

# Convert the target robustly whether Kaggle stores it as Yes/No or already as 0/1.
if not pd.api.types.is_numeric_dtype(train[TARGET]):
    train[TARGET] = (
        train[TARGET]
        .astype(str)
        .str.strip()
        .str.lower()
        .map({
            "no": 0.0,
            "yes": 1.0,
            "0": 0.0,
            "1": 1.0,
            "false": 0.0,
            "true": 1.0,
        })
    )
else:
    train[TARGET] = pd.to_numeric(train[TARGET], errors="coerce")

if train[TARGET].isna().any():
    raise ValueError("Target conversion produced missing values.")

train[TARGET] = train[TARGET].astype(np.float32)
y = train[TARGET].to_numpy(dtype=np.float32)

print("Train:", train.shape)
print("Test :", test.shape)
print("Positive rate:", float(y.mean()))


# ## Fold-safe feature engineering
# 
# The historical record representation is rebuilt independently inside every outer fold.

# ==========================================================================================
# Fold-safe record feature engineering
# ==========================================================================================

def safe_numeric(series):
    """Convert a column to float while preserving missing values."""
    return pd.to_numeric(series, errors="coerce").astype(np.float64)


def stringify_values(values):
    """Convert arbitrary values into stable categorical strings."""
    s = pd.Series(values, copy=False).astype("object")
    s = s.where(~pd.isna(s), "__NA__")
    return s.astype(str).to_numpy(dtype=object)


def build_base_frame(df):
    """
    Build the 67 base numerical features:
      - 7 raw numerics
      - 56 extracted decimal digits
      - 4 domain flags
    """
    out = pd.DataFrame(index=np.arange(len(df)))

    for col in RAW_NUMS:
        out[col] = safe_numeric(df[col]).to_numpy()

    for col in RAW_NUMS:
        x = safe_numeric(df[col]).to_numpy()

        for k in DIGIT_KS:
            scale = 10.0 ** k
            digit = np.floor(x / scale)
            digit = np.mod(digit, 10.0)
            out[f"{col}__digit_k{k}"] = digit

    income = safe_numeric(df["Annual_Income_USD"]).to_numpy()
    concern = safe_numeric(df["Environmental_Concern_Level"]).to_numpy()

    out["flag_income_eq_30000"] = (income == 30000).astype(np.float64)
    out["flag_income_ge_170537"] = (income >= 170537).astype(np.float64)
    out["flag_income_38k_42k"] = (
        (income >= 38000) & (income <= 42000)
    ).astype(np.float64)
    out["flag_env_concern_eq_1"] = (concern == 1).astype(np.float64)

    return out


def fill_base_from_train(train_base, valid_base, test_base):
    """Median-impute base features using outer-fold training rows only."""
    for col in train_base.columns:
        median = np.nanmedian(train_base[col].to_numpy(dtype=np.float64))

        if not np.isfinite(median):
            median = 0.0

        train_base[col] = train_base[col].fillna(median)
        valid_base[col] = valid_base[col].fillna(median)
        test_base[col] = test_base[col].fillna(median)


def source_values(raw_df, base_df, source_name):
    """Return one of the 69 categorical sources used for frequency/target encoding."""
    kind, col = source_name.split("::", 1)

    if kind == "rawcat":
        return stringify_values(raw_df[col].to_numpy())

    if kind == "numcat":
        return stringify_values(safe_numeric(raw_df[col]).to_numpy())

    if kind == "digitcat":
        return stringify_values(base_df[col].to_numpy())

    raise ValueError(source_name)


def make_single_col_matrix(values):
    return np.asarray(values, dtype=object).reshape(-1, 1)


def target_encode_builtin(
    train_values,
    valid_values,
    test_values,
    y_train,
    smooth,
    seed,
):
    """
    Fold-safe sklearn TargetEncoder.

    fit_transform() produces OOF values for the current outer-fold training rows.
    transform() is then applied to outer validation and competition test rows.
    """
    encoder = TargetEncoder(
        smooth=smooth,
        cv=5,
        shuffle=True,
        random_state=seed,
        target_type="binary",
    )

    train_encoded = encoder.fit_transform(
        make_single_col_matrix(train_values),
        y_train,
    ).reshape(-1)

    valid_encoded = encoder.transform(
        make_single_col_matrix(valid_values)
    ).reshape(-1)

    test_encoded = encoder.transform(
        make_single_col_matrix(test_values)
    ).reshape(-1)

    return train_encoded, valid_encoded, test_encoded


def build_fold_features(train_raw, test_raw, train_idx, valid_idx, fold_number):
    """
    Rebuild the complete record representation independently for one outer fold.

    Expected Fold-1 accounting:
        67 base
      + 69 frequency
      + 69 TE(auto)
      + 69 TE(10)
      = 274 numerical candidates

    Historical constant removal:
        274 - 76 = 198 numerical features

    Plus 6 raw categorical IDs:
        198 + 6 = 204 final features
    """
    fold_train = train_raw.iloc[train_idx].reset_index(drop=True)
    fold_valid = train_raw.iloc[valid_idx].reset_index(drop=True)

    y_train = fold_train[TARGET].to_numpy(dtype=np.float32)
    y_valid = fold_valid[TARGET].to_numpy(dtype=np.float32)

    base_train = build_base_frame(fold_train)
    base_valid = build_base_frame(fold_valid)
    base_test = build_base_frame(test_raw)

    fill_base_from_train(base_train, base_valid, base_test)

    digit_cols = [
        col for col in base_train.columns
        if "__digit_k" in col
    ]

    encoding_sources = (
        [f"rawcat::{col}" for col in RAW_CATS]
        + [f"numcat::{col}" for col in RAW_NUMS]
        + [f"digitcat::{col}" for col in digit_cols]
    )

    assert len(encoding_sources) == 69

    # ------------------------------------------------------------------
    # Frequency encoding
    # ------------------------------------------------------------------

    freq_train = pd.DataFrame(index=np.arange(len(fold_train)))
    freq_valid = pd.DataFrame(index=np.arange(len(fold_valid)))
    freq_test = pd.DataFrame(index=np.arange(len(test_raw)))

    for source in encoding_sources:
        train_values = source_values(fold_train, base_train, source)
        valid_values = source_values(fold_valid, base_valid, source)
        test_values = source_values(test_raw, base_test, source)

        frequency_map = (
            pd.Series(train_values)
            .value_counts(dropna=False, normalize=True)
            .to_dict()
        )

        name = f"freq::{source}"

        freq_train[name] = (
            pd.Series(train_values)
            .map(frequency_map)
            .fillna(0.0)
            .to_numpy(np.float64)
        )

        freq_valid[name] = (
            pd.Series(valid_values)
            .map(frequency_map)
            .fillna(0.0)
            .to_numpy(np.float64)
        )

        freq_test[name] = (
            pd.Series(test_values)
            .map(frequency_map)
            .fillna(0.0)
            .to_numpy(np.float64)
        )

    # ------------------------------------------------------------------
    # Raw categorical IDs
    # ------------------------------------------------------------------

    catid_train = pd.DataFrame(index=np.arange(len(fold_train)))
    catid_valid = pd.DataFrame(index=np.arange(len(fold_valid)))
    catid_test = pd.DataFrame(index=np.arange(len(test_raw)))

    for col in RAW_CATS:
        train_values = stringify_values(fold_train[col].to_numpy())
        valid_values = stringify_values(fold_valid[col].to_numpy())
        test_values = stringify_values(test_raw[col].to_numpy())

        unique_values = pd.Index(pd.unique(train_values))
        mapping = {
            value: i + 1
            for i, value in enumerate(unique_values)
        }

        name = f"catid::{col}"

        catid_train[name] = (
            pd.Series(train_values)
            .map(mapping)
            .fillna(0)
            .astype(np.float64)
            .to_numpy()
        )

        catid_valid[name] = (
            pd.Series(valid_values)
            .map(mapping)
            .fillna(0)
            .astype(np.float64)
            .to_numpy()
        )

        catid_test[name] = (
            pd.Series(test_values)
            .map(mapping)
            .fillna(0)
            .astype(np.float64)
            .to_numpy()
        )

    # ------------------------------------------------------------------
    # Two historical target-encoding blocks
    # ------------------------------------------------------------------

    te_auto_train = pd.DataFrame(index=np.arange(len(fold_train)))
    te_auto_valid = pd.DataFrame(index=np.arange(len(fold_valid)))
    te_auto_test = pd.DataFrame(index=np.arange(len(test_raw)))

    te_10_train = pd.DataFrame(index=np.arange(len(fold_train)))
    te_10_valid = pd.DataFrame(index=np.arange(len(fold_valid)))
    te_10_test = pd.DataFrame(index=np.arange(len(test_raw)))

    # Fold 1 uses seed 21, matching the historical fold seed.
    encoder_seed = SEED + (fold_number - 1)

    print(f"Fold {fold_number}: building 2 x 69 target encodings...")

    for j, source in enumerate(encoding_sources):
        train_values = source_values(fold_train, base_train, source)
        valid_values = source_values(fold_valid, base_valid, source)
        test_values = source_values(test_raw, base_test, source)

        auto_train, auto_valid, auto_test = target_encode_builtin(
            train_values,
            valid_values,
            test_values,
            y_train,
            smooth="auto",
            seed=encoder_seed,
        )

        ten_train, ten_valid, ten_test = target_encode_builtin(
            train_values,
            valid_values,
            test_values,
            y_train,
            smooth=10.0,
            seed=encoder_seed,
        )

        te_auto_train[f"te_auto::{source}"] = auto_train
        te_auto_valid[f"te_auto::{source}"] = auto_valid
        te_auto_test[f"te_auto::{source}"] = auto_test

        te_10_train[f"te_10::{source}"] = ten_train
        te_10_valid[f"te_10::{source}"] = ten_valid
        te_10_test[f"te_10::{source}"] = ten_test

        if (j + 1) % 10 == 0 or j + 1 == len(encoding_sources):
            print(f"  encoded {j + 1}/{len(encoding_sources)} sources")

    numeric_train = pd.concat(
        [
            base_train.reset_index(drop=True),
            freq_train,
            te_auto_train,
            te_10_train,
        ],
        axis=1,
    )

    numeric_valid = pd.concat(
        [
            base_valid.reset_index(drop=True),
            freq_valid,
            te_auto_valid,
            te_10_valid,
        ],
        axis=1,
    )

    numeric_test = pd.concat(
        [
            base_test.reset_index(drop=True),
            freq_test,
            te_auto_test,
            te_10_test,
        ],
        axis=1,
    )

    assert numeric_train.shape[1] == 274, numeric_train.shape

    numeric_train = numeric_train.replace([np.inf, -np.inf], np.nan)
    numeric_valid = numeric_valid.replace([np.inf, -np.inf], np.nan)
    numeric_test = numeric_test.replace([np.inf, -np.inf], np.nan)

    # Final train-only median imputation.
    for col in numeric_train.columns:
        median = np.nanmedian(
            numeric_train[col].to_numpy(dtype=np.float64)
        )

        if not np.isfinite(median):
            median = 0.0

        numeric_train[col] = numeric_train[col].fillna(median)
        numeric_valid[col] = numeric_valid[col].fillna(median)
        numeric_test[col] = numeric_test[col].fillna(median)

    # Remove numerical columns that are constant in this fold's training rows.
    unique_counts = numeric_train.nunique(dropna=False)

    keep_numeric_cols = unique_counts[
        unique_counts > 1
    ].index.tolist()

    dropped_numeric_cols = unique_counts[
        unique_counts <= 1
    ].index.tolist()

    numeric_train = numeric_train[keep_numeric_cols]
    numeric_valid = numeric_valid[keep_numeric_cols]
    numeric_test = numeric_test[keep_numeric_cols]

    full_train = pd.concat(
        [
            numeric_train.reset_index(drop=True),
            catid_train,
        ],
        axis=1,
    )

    full_valid = pd.concat(
        [
            numeric_valid.reset_index(drop=True),
            catid_valid,
        ],
        axis=1,
    )

    full_test = pd.concat(
        [
            numeric_test.reset_index(drop=True),
            catid_test,
        ],
        axis=1,
    )

    feature_names = full_train.columns.tolist()

    print(
        f"Fold {fold_number}: "
        f"{len(keep_numeric_cols)} numerical + "
        f"{catid_train.shape[1]} categorical IDs = "
        f"{len(feature_names)} final features"
    )

    if fold_number == 1:
        print(
            "Historical Fold-1 target: "
            "198 numerical + 6 categorical IDs = 204."
        )

        if len(feature_names) != 204:
            print(
                "WARNING: Fold 1 did not reproduce 204 features. "
                "Do not treat this run as an exact record-feature reproduction."
            )

        print(
            f"Fold 1 numerical constants removed: "
            f"{len(dropped_numeric_cols)}"
        )

    x_train_raw = full_train.to_numpy(dtype=np.float32)
    x_valid_raw = full_valid.to_numpy(dtype=np.float32)
    x_test_raw = full_test.to_numpy(dtype=np.float32)

    # Standardize from outer-fold training rows only.
    mean = x_train_raw.mean(
        axis=0,
        dtype=np.float64,
    ).astype(np.float32)

    std = x_train_raw.std(
        axis=0,
        dtype=np.float64,
    ).astype(np.float32)

    mean = np.where(
        np.isfinite(mean),
        mean,
        0.0,
    ).astype(np.float32)

    std = np.where(
        np.isfinite(std) & (std > 1e-7),
        std,
        1.0,
    ).astype(np.float32)

    x_train = np.clip(
        (x_train_raw - mean) / std,
        -8.0,
        8.0,
    ).astype(np.float32)

    x_valid = np.clip(
        (x_valid_raw - mean) / std,
        -8.0,
        8.0,
    ).astype(np.float32)

    x_test = np.clip(
        (x_test_raw - mean) / std,
        -8.0,
        8.0,
    ).astype(np.float32)

    metadata = pd.DataFrame({
        "feature_name": feature_names,
        "mean": mean,
        "std": std,
    })

    return (
        x_train,
        y_train,
        x_valid,
        y_valid,
        x_test,
        feature_names,
        metadata,
    )



# ==========================================================================================
# Experiment 2: Stochastic RPU boosting with hard Newton leaves
# ==========================================================================================

RELATION_POOL = (
    "signed_sq",
    "elementwise_product",
    "normalized_diff",
    "soft_ratio",
    "projection",
    "orthogonal_residual",
)

BOOST_STAGES = 75
SCORER_EPOCHS = 2
N_RELATIONS_PER_STAGE = 3
ROW_FRAC_RANGE = (0.75, 0.90)
FEATURE_FRAC_RANGE = (0.65, 0.80)
N_HARD_LEAVES = 8
MIN_LEAF_ROWS = 3000
NEWTON_L2 = 10.0
BOOST_LR = 0.05
SCORER_LR = 1.5e-3
SCORER_WEIGHT_DECAY = 2e-5


class StochasticRPUScorer(nn.Module):
    """Short-lived RPU scorer used to create a 1D ordering for one boosting stage."""

    def __init__(self, input_dim, relations):
        super().__init__()
        self.input_dim = int(input_dim)
        self.relations = tuple(relations)
        self.joint_dim = self.input_dim * len(self.relations)

        knot_x = torch.linspace(GLOBAL_SPLINE_MIN, GLOBAL_SPLINE_MAX, HERMITE_KNOTS)
        self.register_buffer("knot_x", knot_x)
        self.spline_y = nn.Parameter(knot_x.unsqueeze(0).repeat(self.input_dim, 1).clone())
        self.center = nn.Parameter(torch.randn(self.input_dim) * 0.15)

        scale1 = 0.20 / math.sqrt(max(self.joint_dim, 1))
        scale2 = 0.20 / math.sqrt(max(PATTERN_HIDDEN, 1))
        self.pattern_w1 = nn.Parameter(torch.randn(PATTERN_HIDDEN, self.joint_dim) * scale1)
        self.pattern_b1 = nn.Parameter(torch.zeros(PATTERN_HIDDEN))
        self.pattern_w2 = nn.Parameter(torch.randn(PATTERN_HIDDEN) * scale2)
        self.pattern_b2 = nn.Parameter(torch.zeros(()))

        inv_softplus_one = math.log(math.expm1(1.0))
        self.raw_fire_increments = nn.Parameter(
            torch.full((N_FIRE_KNOTS - 1,), inv_softplus_one)
        )
        self.raw_score_scale = nn.Parameter(torch.tensor(math.log(math.expm1(2.0))))
        self.score_bias = nn.Parameter(torch.zeros(()))

    def spline_transform(self, x):
        lo = float(self.knot_x[0])
        hi = float(self.knot_x[-1])
        n = int(self.knot_x.numel())
        xc = x.clamp(lo, hi)
        pos = (xc - lo) / max(hi - lo, 1e-8) * (n - 1)
        i0 = torch.floor(pos).long().clamp(0, n - 2)
        i1 = i0 + 1
        t = pos - i0.to(pos.dtype)

        h = (hi - lo) / max(n - 1, 1)
        y = self.spline_y
        m = torch.empty_like(y)
        m[:, 0] = (y[:, 1] - y[:, 0]) / h
        m[:, -1] = (y[:, -1] - y[:, -2]) / h
        m[:, 1:-1] = (y[:, 2:] - y[:, :-2]) / (2.0 * h)

        ytab = y.unsqueeze(0).expand(x.shape[0], -1, -1)
        mtab = m.unsqueeze(0).expand(x.shape[0], -1, -1)
        y0 = torch.gather(ytab, 2, i0.unsqueeze(2)).squeeze(2)
        y1 = torch.gather(ytab, 2, i1.unsqueeze(2)).squeeze(2)
        m0 = torch.gather(mtab, 2, i0.unsqueeze(2)).squeeze(2)
        m1 = torch.gather(mtab, 2, i1.unsqueeze(2)).squeeze(2)

        t2 = t * t
        t3 = t2 * t
        h00 = 2.0 * t3 - 3.0 * t2 + 1.0
        h10 = t3 - 2.0 * t2 + t
        h01 = -2.0 * t3 + 3.0 * t2
        h11 = t3 - t2
        return h00 * y0 + h10 * h * m0 + h01 * y1 + h11 * h * m1

    def relation_block(self, z):
        c = self.center.unsqueeze(0)
        diff = z - c

        dot = (z * c).sum(dim=1, keepdim=True)
        c_norm_sq = c.pow(2).sum(dim=1, keepdim=True).clamp_min(1e-6)
        projection = (dot / c_norm_sq) * c
        orthogonal = z - projection

        blocks = {
            "signed_sq": diff * diff.abs(),
            "elementwise_product": z * c,
            "normalized_diff": diff / (z.abs() + c.abs() + 1e-4),
            "soft_ratio": (z / (c + 1e-4)) / (1.0 + (z / (c + 1e-4)).abs()),
            "projection": projection,
            "orthogonal_residual": orthogonal,
        }
        return torch.cat([blocks[name] for name in self.relations], dim=1)

    def fire_values(self):
        inc = F.softplus(self.raw_fire_increments) + 1e-6
        cum = torch.cumsum(inc, dim=0)
        cum = cum / cum[-1].clamp_min(1e-8)
        return torch.cat([torch.zeros(1, device=cum.device, dtype=cum.dtype), cum])

    def apply_fire_curve(self, score):
        pos = score.clamp(0.0, 1.0) * (N_FIRE_KNOTS - 1)
        i0 = torch.floor(pos).long().clamp(0, N_FIRE_KNOTS - 2)
        frac = pos - i0.to(pos.dtype)
        values = self.fire_values()
        return values[i0] + frac * (values[i0 + 1] - values[i0])

    def pattern_score(self, x):
        z = self.spline_transform(x)
        relations = self.relation_block(z)
        hidden = F.gelu(relations @ self.pattern_w1.t() + self.pattern_b1)
        return torch.sigmoid(hidden @ self.pattern_w2 + self.pattern_b2)

    def scalar_score(self, x):
        return self.apply_fire_curve(self.pattern_score(x))

    def forward(self, x):
        score = self.scalar_score(x)
        scale = F.softplus(self.raw_score_scale) + 1e-4
        return self.score_bias + scale * (score - 0.5)

    def regularization(self):
        spline_d2 = self.spline_y[:, 2:] - 2.0 * self.spline_y[:, 1:-1] + self.spline_y[:, :-2]
        fire = self.fire_values()
        fire_d2 = fire[2:] - 2.0 * fire[1:-1] + fire[:-2]
        return (
            LAMBDA_HERMITE_SMOOTH * spline_d2.pow(2).mean()
            + LAMBDA_FIRE_SMOOTH * fire_d2.pow(2).mean()
        )

    @torch.no_grad()
    def initialize_center(self, x_train, y_train, seed):
        rng = np.random.default_rng(seed)
        positives = np.flatnonzero(y_train >= 0.5)
        pool = positives if len(positives) else np.arange(len(y_train))
        idx = int(rng.choice(pool))
        row = torch.from_numpy(x_train[idx:idx + 1]).to(self.center.device, self.center.dtype)
        self.center.copy_(self.spline_transform(row).squeeze(0))


class IndexedDataset(Dataset):
    def __init__(self, x, y, indices):
        self.x = x
        self.y = y.astype(np.float32)
        self.indices = np.asarray(indices, dtype=np.int64)

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, i):
        idx = int(self.indices[i])
        return torch.from_numpy(self.x[idx]), torch.tensor(self.y[idx], dtype=torch.float32), idx


@torch.no_grad()
def predict_score(model, x, feature_idx, batch_size=PRED_BATCH_SIZE):
    model.eval()
    out = np.empty(len(x), dtype=np.float32)
    for start in range(0, len(x), batch_size):
        end = min(start + batch_size, len(x))
        xb = torch.from_numpy(x[start:end, feature_idx]).to(DEVICE, non_blocking=True)
        out[start:end] = model.scalar_score(xb).float().cpu().numpy()
    return out


def sigmoid_np(x):
    x = np.clip(x, -30.0, 30.0)
    return 1.0 / (1.0 + np.exp(-x))


def bce_from_logits_np(logits, y):
    return float(F.binary_cross_entropy_with_logits(
        torch.from_numpy(logits.astype(np.float32)),
        torch.from_numpy(y.astype(np.float32)),
    ).item())


def leaf_gain(g_sum, h_sum, reg_lambda):
    return 0.5 * float(g_sum * g_sum) / float(h_sum + reg_lambda)


def best_split_for_segment(score_sorted, g_prefix, h_prefix, lo, hi, min_leaf, reg_lambda, max_candidates=512):
    if hi - lo < 2 * min_leaf:
        return None

    positions = np.arange(lo + min_leaf, hi - min_leaf + 1, dtype=np.int64)
    if len(positions) > max_candidates:
        pick = np.linspace(0, len(positions) - 1, max_candidates).round().astype(np.int64)
        positions = positions[pick]

    total_g = g_prefix[hi] - g_prefix[lo]
    total_h = h_prefix[hi] - h_prefix[lo]
    parent_gain = leaf_gain(total_g, total_h, reg_lambda)
    best = None

    for pos in positions:
        if score_sorted[pos - 1] >= score_sorted[pos]:
            continue
        gl = g_prefix[pos] - g_prefix[lo]
        hl = h_prefix[pos] - h_prefix[lo]
        gr = total_g - gl
        hr = total_h - hl
        gain = leaf_gain(gl, hl, reg_lambda) + leaf_gain(gr, hr, reg_lambda) - parent_gain
        if best is None or gain > best[0]:
            best = (gain, int(pos))

    return best


def fit_hard_newton_tree_1d(scores, current_logits, y, n_leaves, min_leaf, reg_lambda):
    order = np.argsort(scores, kind="mergesort")
    score_sorted = scores[order].astype(np.float64)
    p = sigmoid_np(current_logits).astype(np.float64)
    g = (p - y).astype(np.float64)[order]
    h = (p * (1.0 - p)).astype(np.float64)[order]

    g_prefix = np.zeros(len(g) + 1, dtype=np.float64)
    h_prefix = np.zeros(len(h) + 1, dtype=np.float64)
    np.cumsum(g, out=g_prefix[1:])
    np.cumsum(h, out=h_prefix[1:])

    segments = [(0, len(score_sorted))]
    while len(segments) < n_leaves:
        best = None
        best_segment = None
        for j, (lo, hi) in enumerate(segments):
            candidate = best_split_for_segment(
                score_sorted, g_prefix, h_prefix, lo, hi, min_leaf, reg_lambda
            )
            if candidate is not None and (best is None or candidate[0] > best[0]):
                best = candidate
                best_segment = j

        if best is None or best[0] <= 0.0:
            break

        _, pos = best
        lo, hi = segments.pop(best_segment)
        segments.extend([(lo, pos), (pos, hi)])
        segments.sort(key=lambda pair: pair[0])

    thresholds = []
    leaf_values = []
    leaf_sizes = []

    for j, (lo, hi) in enumerate(segments):
        g_sum = g_prefix[hi] - g_prefix[lo]
        h_sum = h_prefix[hi] - h_prefix[lo]
        leaf_values.append(-float(g_sum) / float(h_sum + reg_lambda))
        leaf_sizes.append(hi - lo)
        if j < len(segments) - 1:
            next_lo = segments[j + 1][0]
            thresholds.append(0.5 * (float(score_sorted[next_lo - 1]) + float(score_sorted[next_lo])))

    return (
        np.asarray(thresholds, dtype=np.float32),
        np.asarray(leaf_values, dtype=np.float32),
        np.asarray(leaf_sizes, dtype=np.int64),
    )


def apply_hard_leaf_model(scores, thresholds, leaf_values):
    return leaf_values[np.searchsorted(thresholds, scores, side="right")]


def train_fold1(train_raw, test_raw, train_idx, valid_idx):
    print("\n" + "=" * 120)
    print(f"STOCHASTIC RPU BOOSTING | FOLD 1/{N_SPLITS} | train={len(train_idx):,} | valid={len(valid_idx):,}")
    print("=" * 120)

    x_train, y_train, x_valid, y_valid, x_test, feature_names, _ = build_fold_features(
        train_raw, test_raw, train_idx, valid_idx, fold_number=1
    )

    n_features = x_train.shape[1]
    prior = float(np.clip(y_train.mean(), 1e-6, 1.0 - 1e-6))
    prior_logit = float(np.log(prior / (1.0 - prior)))
    train_logit = np.full(len(x_train), prior_logit, dtype=np.float32)
    valid_logit = np.full(len(x_valid), prior_logit, dtype=np.float32)
    test_logit = np.full(len(x_test), prior_logit, dtype=np.float32)

    print(f"Features available     : {n_features}")
    print(f"Feature fraction       : {FEATURE_FRAC_RANGE[0]:.2f}-{FEATURE_FRAC_RANGE[1]:.2f}")
    print(f"Row fraction           : {ROW_FRAC_RANGE[0]:.2f}-{ROW_FRAC_RANGE[1]:.2f}")
    print(f"Relations per stage    : {N_RELATIONS_PER_STAGE}/{len(RELATION_POOL)}")
    print(f"Relation pool          : {', '.join(RELATION_POOL)}")
    print(f"Scorer epochs/stage    : {SCORER_EPOCHS}")
    print(f"Hard leaves/stage      : {N_HARD_LEAVES}")
    print(f"Boost stages           : {BOOST_STAGES}")
    print(f"Boost learning rate    : {BOOST_LR}")

    criterion = nn.BCEWithLogitsLoss()
    best_auc = float(roc_auc_score(y_valid, sigmoid_np(valid_logit)))
    best_stage = 0
    best_valid_logit = valid_logit.copy()
    best_test_logit = test_logit.copy()
    history = []

    for stage in range(1, BOOST_STAGES + 1):
        stage_seed = SEED + 10000 + stage
        rng = np.random.default_rng(stage_seed)
        random.seed(stage_seed)
        torch.manual_seed(stage_seed)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(stage_seed)

        feature_frac = float(rng.uniform(*FEATURE_FRAC_RANGE))
        row_frac = float(rng.uniform(*ROW_FRAC_RANGE))
        n_stage_features = max(2, int(round(feature_frac * n_features)))
        n_stage_rows = max(2 * MIN_LEAF_ROWS, int(round(row_frac * len(x_train))))

        feature_idx = np.sort(rng.choice(n_features, size=n_stage_features, replace=False))
        row_idx = np.sort(rng.choice(len(x_train), size=n_stage_rows, replace=False))
        relations = tuple(rng.choice(RELATION_POOL, size=N_RELATIONS_PER_STAGE, replace=False).tolist())

        scorer = StochasticRPUScorer(n_stage_features, relations).to(DEVICE)
        scorer.initialize_center(
            x_train[row_idx][:, feature_idx], y_train[row_idx], stage_seed + 1234
        )
        optimizer = torch.optim.AdamW(
            scorer.parameters(), lr=SCORER_LR, weight_decay=SCORER_WEIGHT_DECAY
        )

        dataset = IndexedDataset(x_train[:, feature_idx], y_train, row_idx)
        loader = DataLoader(
            dataset,
            batch_size=BATCH_SIZE,
            shuffle=True,
            num_workers=0,
            pin_memory=(DEVICE.type == "cuda"),
            drop_last=False,
        )

        for _ in range(SCORER_EPOCHS):
            scorer.train()
            for xb, yb, original_idx in loader:
                xb = xb.to(DEVICE, non_blocking=True)
                yb = yb.to(DEVICE, non_blocking=True)
                base = torch.from_numpy(train_logit[original_idx.numpy()]).to(DEVICE)

                optimizer.zero_grad(set_to_none=True)
                candidate = base + scorer(xb)
                loss = criterion(candidate, yb) + scorer.regularization()
                loss.backward()
                torch.nn.utils.clip_grad_norm_(scorer.parameters(), GRAD_CLIP)
                optimizer.step()

        train_score = predict_score(scorer, x_train, feature_idx)
        valid_score = predict_score(scorer, x_valid, feature_idx)
        test_score = predict_score(scorer, x_test, feature_idx)

        # Fit hard Newton leaves only on the stage's sampled training rows.
        thresholds, leaf_values, leaf_sizes = fit_hard_newton_tree_1d(
            train_score[row_idx],
            train_logit[row_idx],
            y_train[row_idx],
            n_leaves=N_HARD_LEAVES,
            min_leaf=MIN_LEAF_ROWS,
            reg_lambda=NEWTON_L2,
        )

        train_step = BOOST_LR * apply_hard_leaf_model(train_score, thresholds, leaf_values)
        valid_step = BOOST_LR * apply_hard_leaf_model(valid_score, thresholds, leaf_values)
        test_step = BOOST_LR * apply_hard_leaf_model(test_score, thresholds, leaf_values)

        prev_auc = float(roc_auc_score(y_valid, sigmoid_np(valid_logit)))
        train_logit += train_step
        valid_logit += valid_step
        test_logit += test_step

        auc = float(roc_auc_score(y_valid, sigmoid_np(valid_logit)))
        valid_bce = bce_from_logits_np(valid_logit, y_valid)
        auc_gain = auc - prev_auc
        improved = auc > best_auc + 1e-7

        if improved:
            best_auc = auc
            best_stage = stage
            best_valid_logit = valid_logit.copy()
            best_test_logit = test_logit.copy()

        history.append({
            "stage": stage,
            "valid_auc": auc,
            "auc_gain": auc_gain,
            "valid_bce": valid_bce,
            "row_fraction": row_frac,
            "feature_fraction": feature_frac,
            "n_features": n_stage_features,
            "relations": ",".join(relations),
            "n_leaves": len(leaf_values),
            "mean_abs_step": float(np.mean(np.abs(valid_step))),
            "min_leaf_rows": int(leaf_sizes.min()),
            "max_leaf_rows": int(leaf_sizes.max()),
        })

        marker = " *BEST*" if improved else ""
        print(
            f"Stage {stage:02d}/{BOOST_STAGES} | "
            f"rows={row_frac:.2f} | feats={n_stage_features:03d} ({feature_frac:.2f}) | "
            f"rels={'+'.join(relations)} | AUC={auc:.8f} | gain={auc_gain:+.8f} | "
            f"BCE={valid_bce:.6f} | mean|step|={np.mean(np.abs(valid_step)):.4f}{marker}"
        )

        del scorer, optimizer, loader, dataset
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    pd.DataFrame(history).to_csv(
        os.path.join(OUTPUT_DIR, "rpu_experiment2_stochastic_boosting_history.csv"), index=False
    )

    valid_pred = sigmoid_np(best_valid_logit)
    test_pred = sigmoid_np(best_test_logit)
    pd.DataFrame({"row": valid_idx, TARGET: valid_pred}).to_csv(
        os.path.join(OUTPUT_DIR, "rpu_experiment2_fold1_valid_predictions.csv"), index=False
    )

    if sample_path is not None:
        submission = pd.read_csv(sample_path)
        submission[TARGET] = test_pred
        submission.to_csv(
            os.path.join(OUTPUT_DIR, "rpu_experiment2_fold1_test_preview.csv"), index=False
        )

    print("-" * 120)
    print(f"BEST STAGE : {best_stage}")
    print(f"BEST AUC   : {best_auc:.8f}")
    print("-" * 120)
    return best_auc


outer_cv = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
train_idx, valid_idx = next(outer_cv.split(np.zeros(len(train)), y))
auc = train_fold1(train, test, train_idx, valid_idx)

print("\n" + "=" * 120)
print("EXPERIMENT 2 — STOCHASTIC RPU BOOSTING COMPLETE")
print(f"Validation AUC: {auc:.8f}")
print("=" * 120)
